# Add Indices

## Import libs

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import spyndex
import rasterio
from pathlib import Path
from IPython.display import display, HTML

## Define File Paths

In [ ]:
notebook_dir = Path.cwd()
base_path = notebook_dir.parent.parent / "Data" 
base_path_str = str(base_path)
print(f"Base path automatically set to: {base_path_str}")

In [ ]:
cropped_dir = os.path.join(base_path, "6_Cropped_Images")
out_dir = os.path.join(base_path, "8_Images_with_Indices")
os.makedirs(out_dir, exist_ok=True)

## Define Functions

In [ ]:
# Robust scaling using 1 and 99 percentile
def scale_to_01(array, valid_mask):
    valid_data = array[valid_mask]
    valid_data = valid_data[np.isfinite(valid_data)] 

    scaled = np.full_like(array, fill_value=np.nan, dtype=np.float32)
    
    if len(valid_data) == 0:
        return scaled
        
    vmin, vmax = np.percentile(valid_data, 1), np.percentile(valid_data, 99)
        
    if vmax - vmin < 1e-6:
        scaled[valid_mask] = 0.0
    else:
        scaled[valid_mask] = np.clip((array[valid_mask] - vmin) / (vmax - vmin), 0.0, 1.0)
        
    return scaled

## Add indices and scale all bands

In [ ]:
tif_files = glob.glob(os.path.join(cropped_dir, "*.tif"))
print(f"Found TIFs for processing: {len(tif_files)}\n")

for input_path in tif_files:
    filename = os.path.basename(input_path)
    print(f"⏳ Processing {filename}...")
    
    # 1. Read image
    with rasterio.open(input_path) as src:
        meta = src.meta.copy()
        data = src.read() # Shape: (4, Height, Width) -> R, G, B, CHM
        valid_mask_2d = src.dataset_mask()
        
    nodata_mask = (valid_mask_2d == 0)
    valid_mask = ~nodata_mask
    
    # 2. Prepare RGB for index calulcation
    r = (data[0] / 255.0).astype(np.float32)
    g = (data[1] / 255.0).astype(np.float32)
    b = (data[2] / 255.0).astype(np.float32)
        
    r[nodata_mask] = np.nan
    g[nodata_mask] = np.nan
    b[nodata_mask] = np.nan

    print("R: ",r[~nodata_mask].min(), r[~nodata_mask].mean(), r[~nodata_mask].max())
    print("G: ",g[~nodata_mask].min(), g[~nodata_mask].mean(), g[~nodata_mask].max())
    print("B: ",b[~nodata_mask].min(), b[~nodata_mask].mean(), b[~nodata_mask].max())
    
    # 3. Calculate Indices
    import warnings
    with warnings.catch_warnings():
        warnings.simplefilter("ignore") 
        idx = spyndex.computeIndex(
            index=["ExG", "VARI", "NGRDI"],
            params={"R": r, "G": g, "B": b}
        )

    exg = idx[0]
    vari = idx[1]
    ngrdi = idx[2]
    print("ExG:   ", np.nanmin(exg), np.nanmean(exg), np.nanmax(exg))
    print("VARI:  ", np.nanmin(vari), np.nanmean(vari), np.nanmax(vari))
    print("NGRDI: ", np.nanmin(ngrdi), np.nanmean(ngrdi), np.nanmax(ngrdi))

    # 4. Scale indices
    exg_scaled = scale_to_01(idx[0], valid_mask)
    vari_scaled = scale_to_01(idx[1], valid_mask)
    ngrdi_scaled = scale_to_01(idx[2], valid_mask)

    print("ExG:   ", np.nanmin(exg_scaled[~nodata_mask]), np.nanmean(exg_scaled[~nodata_mask]), np.nanmax(exg_scaled[~nodata_mask]))
    print("VARI:  ", np.nanmin(vari_scaled[~nodata_mask]), np.nanmean(vari_scaled[~nodata_mask]), np.nanmax(vari_scaled[~nodata_mask]))
    print("NGRDI: ", np.nanmin(ngrdi_scaled[~nodata_mask]), np.nanmean(ngrdi_scaled[~nodata_mask]), np.nanmax(ngrdi_scaled[~nodata_mask]))
    
    # 5. Prepare array for export
    out_bands = []
    
    # Add Bands 1-4 (R, G, B, CHM) and set NoData to -9999.0 
    for i in range(4):
        band = data[i].astype(np.float32)

        # Scale RGB to [0,1]
        if i < 3: 
            band = band / 255.0

        # Scale CHM (Index 3) to [0,1] 
        elif i == 3:
            band = scale_to_01(band, valid_mask)
            
        band[nodata_mask] = -9999.0
        out_bands.append(band)
        
    # Add Bands 5-7 (ExG, VARI, NGRDI) and set NoData to -9999.0 
    for scaled_index in [exg_scaled, vari_scaled, ngrdi_scaled]:
        scaled_index[nodata_mask] = -9999.0
        out_bands.append(scaled_index)
        
    # Stack all 7 bands -> Shape: (7, Height, Width)
    out_array = np.stack(out_bands)
    
    # 6. Adapt Metadata
    meta.update({
        "count": 7,
        "dtype": "float32",
        "nodata": -9999.0
    })
    
    # 7. Save
    out_filename = filename.replace("_resampled_cropped", "_7_channel").replace("_cropped", "_7_channel")
    if out_filename == filename: 
        out_filename = filename.replace(".tif", "_7_channel.tif")
        
    output_path = os.path.join(out_dir, out_filename)
    
    with rasterio.open(output_path, "w", **meta) as dest:
        dest.write(out_array)
        
    print(f"  ✅ Saved as: {out_filename}")

print("\n🚀 BATCH-PROCESSING FINISHED!")

## Check scales

In [ ]:
indices_dir = os.path.join(base_path, "8_Images_with_Indices")
ML_NODATA = -9999

for image in os.listdir(indices_dir):

    if ".tif" in image:
        image_path = os.path.join(indices_dir, image)

        with rasterio.open(image_path) as src:
            meta = src.meta.copy()
            new_data = src.read() 

        # Create an empty list to collect channel statistics
        stats_list = []
            
        # Loop through the number of channels
        for channel in range(new_data.shape[0]):
            band_data = new_data[channel]
            valid_data = band_data[band_data != ML_NODATA]
        
            if valid_data.size > 0:
                b_min = np.nanmin(valid_data)
                b_mean = np.nanmean(valid_data)
                b_max = np.nanmax(valid_data)
            else:
                b_min, b_mean, b_max = np.nan, np.nan, np.nan
            
            stats_list.append({
                "Band": f"Band {channel+1}",
                "Min": b_min,
                "Mean": b_mean,
                "Max": b_max,
                "Actual NaNs (-9999.0)": np.sum(band_data==ML_NODATA),
                "Zeros (0.0)": np.sum(band_data == 0.0)
            })
        
        # Convert to DataFrame for beautiful notebook rendering
        df_stats = pd.DataFrame(stats_list)
        df_stats.set_index("Band", inplace=True)
        
        # Display 
        print("=========================================")
        print(f"IMAGE: {image}")
        print("=========================================")
        
        display(df_stats.round(4)) 
        print("\n")

        

In [ ]:
# Create an empty list to collect channel statistics
stats_list = []

ML_NODATA = -9999

# Loop through the number of channels
for channel in range(new_data.shape[0]):
    band_data = new_data[channel]
    valid_data = band_data[band_data != ML_NODATA]

    if valid_data.size > 0:
        b_min = np.nanmin(valid_data)
        b_mean = np.nanmean(valid_data)
        b_max = np.nanmax(valid_data)
    else:
        b_min, b_mean, b_max = np.nan, np.nan, np.nan
    
    stats_list.append({
        "Band": f"Band {channel+1}",
        "Min": b_min,
        "Mean": b_mean,
        "Max": b_max,
        "Actual NaNs (-9999.0)": np.sum(band_data==ML_NODATA),
        "Zeros (0.0)": np.sum(band_data == 0.0)
    })

# Convert to DataFrame for beautiful notebook rendering
df_stats = pd.DataFrame(stats_list)
df_stats.set_index("Band", inplace=True)

# Display the beautiful table
df_stats